# Flatten the model

## Prune ResNet (works!!!)

In [1]:
import torch
import torch.nn as nn
import torch_pruning as tp
from torchvision import models
import random
import torch_pruning as tp
from ultralytics import YOLOv10

# Load a pre-trained ResNet model
model = models.resnet50(pretrained=True)

# Set the model to evaluation mode to prevent any unintended training behavior
#print(model)

# Function to prune the model
def prune_model(model, prune_percentage):
    # Create a dependency graph while the model is in evaluation mode
    DG = tp.DependencyGraph().build_dependency(model, example_inputs=torch.randn(1, 3, 224, 224))

    # Function to prune a single layer
    def prune_layer(layer, amount):
        if isinstance(layer, nn.Conv2d):
            # Generate a pruning plan for pruning output channels
            pruning_group = DG.get_pruning_group(layer, tp.prune_conv_out_channels, idxs=[0,1,2,3])
            # Execute the pruning plan
            pruning_group.prune()

    # Flatten the model to access each layer
    flattened_layers = [module for module in model.modules() if isinstance(module, nn.Conv2d)]

    # Apply pruning to each convolutional layer
    for i, layer in enumerate(flattened_layers):
        print(i, layer)
        amount = random.uniform(0, prune_percentage)
        prune_layer(layer, amount)
        print(layer)

# Apply pruning to the ResNet model with up to 50% pruning per layer
prune_percentage = 0.5
prune_model(model, prune_percentage)

print("pruning is done")
# Ensure the model remains in evaluation mode after pruning
model.eval()

# Print the pruned model to verify
#print(model)


/conda/anaconda3/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/conda/anaconda3/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


0 Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
Conv2d(3, 60, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
1 Conv2d(60, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
Conv2d(60, 60, kernel_size=(1, 1), stride=(1, 1), bias=False)
2 Conv2d(60, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Conv2d(60, 60, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
3 Conv2d(60, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
Conv2d(60, 252, kernel_size=(1, 1), stride=(1, 1), bias=False)
4 Conv2d(60, 252, kernel_size=(1, 1), stride=(1, 1), bias=False)
Conv2d(60, 248, kernel_size=(1, 1), stride=(1, 1), bias=False)
5 Conv2d(248, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
Conv2d(248, 60, kernel_size=(1, 1), stride=(1, 1), bias=False)
6 Conv2d(60, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Conv2d(60, 60, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
7 Conv2d(60, 248, 

ResNet(
  (conv1): Conv2d(3, 60, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(60, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(60, 60, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(60, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(60, 60, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(60, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(60, 240, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(240, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(60, 240, kernel_size=(1, 1), stride=(1, 

# Prune YOLOv10 (works until layer 82)

In [2]:
import torch
import torch.nn as nn
import torch_pruning as tp
from torchvision import models
import random
import torch_pruning as tp
from ultralytics import YOLOv10
import copy

# Load a pre-trained ResNet model
init_model = YOLOv10.from_pretrained('jameslahm/yolov10x')
model = copy.deepcopy(init_model) 
model = model.model.train()


# Set the model to evaluation mode to prevent any unintended training behavior
#print(model)

# Function to prune the model
import time
import gc

def prune_model(model):
    DG = tp.DependencyGraph().build_dependency(model, example_inputs=torch.randn(1, 3, 224, 224))

    def prune_layer(layer, idxs):
        if isinstance(layer, nn.Conv2d):
            pruning_group = DG.get_pruning_group(layer, tp.prune_conv_out_channels, idxs)
            pruning_group.prune()

    flattened_layers = [module for module in model.modules() if isinstance(module, nn.Conv2d)]

    for i, layer in enumerate(flattened_layers):

        idxs=[0,1,2,3]                
        if i == 82 or i == 83 or i == 84:
            idxs = []

        print(f"Pruning layer {i}: {layer}")
        start_time = time.time()

        prune_layer(layer, idxs)
        
        end_time = time.time()
        print(f"Layer {i} pruned in {end_time - start_time:.2f} seconds")
        del layer
        gc.collect()

prune_model(model)

print("pruning is done")
# Ensure the model remains in evaluation mode after pruning
model.eval()

# Print the pruned model to verify
#print(model)

# Clear gradients
model.zero_grad()

# Save the pruned model
torch.save(model, 'model_pruned.pth')

# Load the pruned model
model = torch.load('model_pruned.pth')

print(model)


Pruning layer 0: Conv2d(3, 80, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
Layer 0 pruned in 0.00 seconds
Pruning layer 1: Conv2d(76, 160, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
Layer 1 pruned in 0.01 seconds
Pruning layer 2: Conv2d(156, 160, kernel_size=(1, 1), stride=(1, 1), bias=False)
Layer 2 pruned in 0.00 seconds
Pruning layer 3: Conv2d(396, 160, kernel_size=(1, 1), stride=(1, 1), bias=False)
Layer 3 pruned in 0.01 seconds
Pruning layer 4: Conv2d(80, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Layer 4 pruned in 0.01 seconds
Pruning layer 5: Conv2d(76, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Layer 5 pruned in 0.04 seconds
Pruning layer 6: Conv2d(76, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Layer 6 pruned in 0.01 seconds
Pruning layer 7: Conv2d(76, 76, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Layer 7 pruned in 0.03 seconds
Pruning layer 8: Con

/tmp/ipykernel_426611/1047965495.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('model_pruned.pth')


In [5]:
del model
model = copy.deepcopy(init_model)
model = model.model.train()
prune_model(model)
print(model)

Pruning layer 0: Conv2d(3, 80, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
Layer 0 pruned in 0.00 seconds
Pruning layer 1: Conv2d(76, 160, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
Layer 1 pruned in 0.01 seconds
Pruning layer 2: Conv2d(156, 160, kernel_size=(1, 1), stride=(1, 1), bias=False)
Layer 2 pruned in 0.00 seconds
Pruning layer 3: Conv2d(396, 160, kernel_size=(1, 1), stride=(1, 1), bias=False)
Layer 3 pruned in 0.01 seconds
Pruning layer 4: Conv2d(80, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Layer 4 pruned in 0.01 seconds
Pruning layer 5: Conv2d(76, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Layer 5 pruned in 0.02 seconds
Pruning layer 6: Conv2d(76, 80, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Layer 6 pruned in 0.01 seconds
Pruning layer 7: Conv2d(76, 76, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Layer 7 pruned in 0.03 seconds
Pruning layer 8: Con